<a href="https://colab.research.google.com/github/Ngwijabagabo/Groundwater_Storage_pred/blob/main/Groundwater%20Potential%20Zone%20(GWPZ)%20maps%20-%20Eastern%20Province%2C%20Rwanda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
"""
Groundwater Potential Zone (GWPZ) maps - Eastern Province, Rwanda
------------------------------------------------------------------
* Fig 1 : Current situation, RF + MaxEnt ensemble (stippled where the two models
          disagree by >= 2 classes)
* Fig 2 : CMIP6 scenarios (SSP1-2.6 / SSP2-4.5 / SSP5-8.5 x 2040-2059 / 2080-2099),
          ensemble; optional per-model versions
Suitability classes are draped over a hillshade (+ light contours) so the
eastern topography stays readable, and district boundaries/labels are drawn
with white casings + halos so they stand out on every class colour.

Expected inputs (same grid, EPSG:32735, classes 1..5):
    {RF|MaxEnt}_{Current|SSP1-26|SSP2-45|SSP5-85}_{period}_class.tif
"""
import os
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker
from matplotlib.colors import LightSource, to_rgb
from matplotlib.patches import Patch
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
from scipy.ndimage import gaussian_filter

# ============================ CONFIG ============================
DATA_DIR  = '/content'          # folder holding the *_class.tif files
OUT_DIR   = '/content/figures'
GRID_M    = 100
CRS       = 'EPSG:32735'
GADM_L2   = 'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_RWA_2.json'
DEM_PATH  = '/content/DEM_projected.tif'   # local DEM (any CRS). None -> Copernicus GLO-90 (AWS)

COMBINE        = 'mean'         # 'mean' (rounded mean class) or 'min' (conservative)
DISAGREE_STEP  = 2              # stipple where |RF - MaxEnt| >= this many classes
TOPO_STRENGTH  = 0.70           # 0 = flat colours, 1 = strong relief shading
VERT_EXAG      = 6              # hillshade vertical exaggeration
CONTOUR_STEP   = 100            # m; None to switch off contours
SHOW_STATS     = False          # % High / % Low box on each map (off: it overlaps the map)
PER_MODEL_FIGS = True           # also draw RF-only and MaxEnt-only scenario figures
DPI            = 400

CLASS_NAMES  = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
CLASS_COLORS = ['#d73027', '#fc8d59', '#fee08b', '#66bd63', '#1a9850']
SSP_LABEL    = {'SSP1-26': 'SSP1-2.6', 'SSP2-45': 'SSP2-4.5', 'SSP5-85': 'SSP5-8.5'}
SSPS         = ('SSP1-26', 'SSP2-45', 'SSP5-85')
PERIODS      = ('2040-2059', '2080-2099')
RUNS         = [('Current', '1991-2020')] + [(s, p) for s in SSPS for p in PERIODS]
MODELS       = ('RF', 'MaxEnt')

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'hatch.color': '#1a1a1a', 'hatch.linewidth': 0.5})


# ============================ IO ============================
def rd(name):
    """Read a class raster -> (int array with 0 = nodata, transform, shape)."""
    with rasterio.open(os.path.join(DATA_DIR, name)) as s:
        a = s.read(1).astype('float32')
        if s.nodata is not None:
            a[a == s.nodata] = np.nan
        a = np.nan_to_num(a, nan=0).astype('int16')
        a[(a < 1) | (a > 5)] = 0
        return a, s.transform, a.shape


EAST_DISTRICTS = ['Bugesera', 'Gatsibo', 'Kayonza', 'Kirehe', 'Ngoma', 'Nyagatare', 'Rwamagana']


def load_districts():
    # select by district name: GADM 4.1 stores the province as 'Iburasirazuba',
    # so a NAME_1 'East' filter can silently return nothing
    d2 = gpd.read_file(GADM_L2)
    dist = d2[d2['NAME_2'].str.strip().str.title().isin(EAST_DISTRICTS)].to_crs(CRS)
    if dist.empty:
        raise ValueError(f'No Eastern districts found; NAME_1 values: {sorted(d2.NAME_1.unique())}')
    print(f'  {len(dist)} districts: {", ".join(sorted(dist.NAME_2))}')
    return dist.rename(columns={'NAME_2': 'District'})[['District', 'geometry']]


def load_dem(shape, transform):
    """DEM resampled onto the class grid (bilinear)."""
    dem = np.full(shape, np.nan, 'float32')
    if DEM_PATH:
        sources = [DEM_PATH]
    else:  # Copernicus GLO-90 tiles covering Eastern Province
        base = 'https://copernicus-dem-90m.s3.amazonaws.com/{0}/{0}.tif'
        sources = [base.format(f'Copernicus_DSM_COG_30_{la}_00_{lo}_00_DEM')
                   for la in ('S01', 'S02', 'S03') for lo in ('E029', 'E030')]
    for src_path in sources:
        try:
            with rasterio.open(src_path) as src:
                tmp = np.full(shape, np.nan, 'float32')
                reproject(rasterio.band(src, 1), tmp, src_nodata=src.nodata,
                          dst_transform=transform, dst_crs=CRS, dst_nodata=np.nan,
                          resampling=Resampling.bilinear)
                tmp[(tmp < -500) | (tmp > 9000)] = np.nan   # undeclared nodata (e.g. -32768)
                dem = np.where(np.isnan(dem), tmp, dem)
        except Exception as e:
            print(f'  DEM source skipped ({os.path.basename(src_path)}): {e}')
    if np.isnan(dem).all():
        raise RuntimeError('No DEM data - set DEM_PATH to a local DEM.')
    return dem


# ============================ PROCESSING ============================
def ensemble(rf, mx):
    valid = (rf > 0) & (mx > 0)
    if COMBINE == 'min':
        e = np.minimum(rf, mx)
    else:
        e = np.floor((rf + mx) / 2 + 0.5)          # round half up
    e = np.where(valid, e, 0).astype('int16')
    disagree = valid & (np.abs(rf.astype(int) - mx) >= DISAGREE_STEP)
    return e, disagree


def drape(cls, hs):
    """Class colours blended with hillshade (overlay) -> RGBA, transparent outside."""
    lut = np.vstack([[1, 1, 1]] + [to_rgb(c) for c in CLASS_COLORS])
    rgb = lut[cls]
    # flat ground keeps its true class colour; shaded slopes darken, lit slopes brighten
    flat = np.sin(np.radians(40))                    # hillshade value of flat terrain
    shade = np.clip(1 + TOPO_STRENGTH * 1.4 * (hs - flat), 0.35, 1.15)[..., None]
    rgb = rgb * shade
    alpha = (cls > 0).astype(float)[..., None]
    return np.clip(np.concatenate([rgb, alpha], -1), 0, 1)


# ============================ DRAWING ============================
def km_axes(ax, xlab=True, ylab=True):
    f = mticker.FuncFormatter(lambda v, _: f'{v / 1000:.0f}')
    ax.xaxis.set_major_formatter(f); ax.yaxis.set_major_formatter(f)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(20000))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(20000))
    ax.tick_params(labelsize=7, length=2)
    ax.grid(color='0.6', lw=0.3, ls=':', alpha=0.6, zorder=0)
    ax.set_xlabel('Easting (km)' if xlab else '', fontsize=8)
    ax.set_ylabel('Northing (km)' if ylab else '', fontsize=8)


def draw_panel(ax, cls, ctx, title, disagree=None, label_size=8, stats=SHOW_STATS):
    ax.imshow(drape(cls, ctx['hs']), extent=ctx['extent'], interpolation='nearest', zorder=1)

    if CONTOUR_STEP and np.isfinite(ctx['dem_s']).any():
        lv = np.arange(np.floor(np.nanmin(ctx['dem_s']) / CONTOUR_STEP) * CONTOUR_STEP,
                       np.nanmax(ctx['dem_s']) + CONTOUR_STEP, CONTOUR_STEP)
        ax.contour(ctx['X'], ctx['Y'], ctx['dem_s'], levels=lv, colors='#3b2a1a',
                   linewidths=0.25, alpha=0.45, zorder=2)

    if disagree is not None and disagree.any():
        ax.contourf(ctx['X'], ctx['Y'], disagree.astype(float), levels=[0.5, 1.5],
                    colors='none', hatches=['.....'], zorder=3)

    dist = ctx['dist']
    if dist is not None:
        # white casing under black line -> visible on red and green alike
        dist.boundary.plot(ax=ax, color='white', lw=2.4, zorder=4)
        dist.boundary.plot(ax=ax, color='black', lw=0.9, zorder=5)
        ctx['prov'].boundary.plot(ax=ax, color='white', lw=4.0, zorder=6)
        ctx['prov'].boundary.plot(ax=ax, color='black', lw=2.0, zorder=7)
        for _, r in dist.iterrows():
            p = r.geometry.representative_point()
            ax.text(p.x, p.y, r.District, ha='center', va='center', fontsize=label_size,
                    fontweight='bold', fontstyle='italic', color='black', zorder=8,
                    path_effects=[pe.withStroke(linewidth=3, foreground='white')])

    if stats:
        v = cls[cls > 0]
        hi = 100 * np.isin(v, (4, 5)).mean() if v.size else 0
        lo = 100 * np.isin(v, (1, 2)).mean() if v.size else 0
        ax.text(0.98, 0.02, f'High+V.High {hi:.1f}%\nLow+V.Low {lo:.1f}%',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=6.5, zorder=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.5', lw=0.5, alpha=0.9))

    ax.set_xlim(ctx['extent'][:2]); ax.set_ylim(ctx['extent'][2:])
    ax.set_title(title, loc='left', fontsize=9, fontweight='bold')
    ax.set_aspect('equal')


def north_arrow_scalebar(ax, ctx, km=20):
    x0, x1, y0, y1 = ctx['extent']
    ax.annotate('', xy=(0.93, 0.93), xytext=(0.93, 0.85), xycoords='axes fraction',
                arrowprops=dict(facecolor='black', width=3, headwidth=9), zorder=9)
    ax.text(0.93, 0.94, 'N', transform=ax.transAxes, ha='center', va='bottom',
            fontsize=11, fontweight='bold', zorder=9)
    sx, sy, step = x0 + 0.05 * (x1 - x0), y0 + 0.03 * (y1 - y0), km * 1000 / 4
    for i in range(4):
        ax.add_patch(plt.Rectangle((sx + i * step, sy), step, 0.006 * (y1 - y0),
                                   fc='black' if i % 2 == 0 else 'white', ec='black', lw=0.6, zorder=9))
    for i in (0, 2, 4):
        ax.text(sx + i * step, sy - 0.006 * (y1 - y0), f'{int(i * km / 4)}',
                ha='center', va='top', fontsize=6.5, zorder=9)
    ax.text(sx + 4 * step + step * 0.3, sy - 0.006 * (y1 - y0), 'km', va='top', fontsize=6.5)


def legend(fig, with_stipple, y=0.01):
    h = [Patch(fc=c, ec='0.3', lw=0.5, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
    if with_stipple:
        h.append(Patch(fc='white', ec='0.3', lw=0.5, hatch='.....',
                       label=f'RF/MaxEnt disagree (≥{DISAGREE_STEP} classes)'))
    if CONTOUR_STEP:
        h.append(plt.Line2D([], [], color='#3b2a1a', lw=0.8, label=f'Contour ({CONTOUR_STEP} m)'))
    fig.legend(handles=h, loc='lower center', bbox_to_anchor=(0.5, y), ncol=len(h),
               title='Groundwater Potential Zone', title_fontsize=9, fontsize=8, frameon=True,
               edgecolor='0.3', handlelength=1.6, columnspacing=1.2)


# ============================ FIGURES ============================
def fig_current(ens, dis, ctx):
    fig, ax = plt.subplots(figsize=(7, 9.2))
    draw_panel(ax, ens, ctx, 'GWPZ — Current situation (RF + MaxEnt ensemble, 1991–2020)',
               disagree=dis, label_size=10)
    km_axes(ax); north_arrow_scalebar(ax, ctx)
    legend(fig, with_stipple=True, y=0.0)
    fig.subplots_adjust(bottom=0.11, top=0.96)
    out = os.path.join(OUT_DIR, 'Fig_GWPZ_current_ensemble.png')
    fig.savefig(out, dpi=DPI, bbox_inches='tight'); plt.close(fig); print('saved', out)


def fig_current_models(rf, mx, ctx):
    """Current RF and current MaxEnt side by side."""
    fig, axs = plt.subplots(1, 2, figsize=(12, 8.6), sharex=True, sharey=True)
    for ax, cls, t, first in ((axs[0], rf, '(a) RF — Current situation (1991–2020)', True),
                              (axs[1], mx, '(b) MaxEnt — Current situation (1991–2020)', False)):
        draw_panel(ax, cls, ctx, t, label_size=9)
        km_axes(ax, ylab=first)
    north_arrow_scalebar(axs[1], ctx)
    legend(fig, with_stipple=False, y=0.0)
    fig.subplots_adjust(left=0.07, right=0.99, top=0.96, bottom=0.12, wspace=0.04)
    out = os.path.join(OUT_DIR, 'Fig_GWPZ_current_RF_vs_MaxEnt.png')
    fig.savefig(out, dpi=DPI, bbox_inches='tight'); plt.close(fig); print('saved', out)


def fig_scenarios(maps, dis, ctx, tag):
    """Rows = periods, columns = SSPs."""
    fig, axs = plt.subplots(2, 3, figsize=(13, 12.5), sharex=True, sharey=True)
    letters = iter('abcdef')
    for r, per in enumerate(PERIODS):
        for c, ssp in enumerate(SSPS):
            ax = axs[r, c]
            draw_panel(ax, maps[(ssp, per)], ctx,
                       f'({next(letters)}) {SSP_LABEL[ssp]} · {per.replace("-", "–")}',
                       disagree=None if dis is None else dis[(ssp, per)], label_size=7)
            km_axes(ax, xlab=(r == 1), ylab=(c == 0))
    north_arrow_scalebar(axs[0, 2], ctx)
    fig.suptitle(f'Projected GWPZ — {tag}', fontsize=12, fontweight='bold', y=0.995)
    legend(fig, with_stipple=dis is not None, y=0.0)
    fig.subplots_adjust(left=0.06, right=0.99, top=0.965, bottom=0.08, wspace=0.05, hspace=0.1)
    out = os.path.join(OUT_DIR, f'Fig_GWPZ_scenarios_{tag.split()[0]}.png')
    fig.savefig(out, dpi=DPI, bbox_inches='tight'); plt.close(fig); print('saved', out)


# ============================ MAIN ============================
def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    CL = {}
    for m in MODELS:
        for s, p in RUNS:
            CL[(m, s, p)], TF, SHP = rd(f'{m}_{s}_{p}_class.tif')

    H, W = SHP
    x0, y1 = TF.c, TF.f
    extent = (x0, x0 + W * TF.a, y1 + H * TF.e, y1)
    xs = x0 + (np.arange(W) + 0.5) * TF.a
    ys = y1 + (np.arange(H) + 0.5) * TF.e
    X, Y = np.meshgrid(xs, ys)

    try:
        DIST = load_districts()
        PROV = DIST.dissolve()
        DR = rasterize([(g, i + 1) for i, g in enumerate(DIST.geometry)],
                       out_shape=SHP, transform=TF, fill=0)
        if not (DR > 0).any():
            raise ValueError('district polygons do not overlap the class rasters (check CRS)')
    except Exception as e:
        print('District boundaries unavailable -> no boundaries/labels:', e)
        DIST = PROV = DR = None

    print('Loading DEM ...')
    dem = load_dem(SHP, TF)
    print(f'  DEM range {np.nanmin(dem):.0f}-{np.nanmax(dem):.0f} m, '
          f'{100 * np.isnan(dem).mean():.1f}% of grid without data')
    dem_f = np.where(np.isnan(dem), np.nanmean(dem), dem)
    hs = LightSource(azdeg=315, altdeg=40).hillshade(dem_f, vert_exag=VERT_EXAG, dx=GRID_M, dy=GRID_M)
    inside = (DR > 0) if DR is not None else (CL[('RF',) + RUNS[0]] > 0)
    dem_s = np.where(inside, gaussian_filter(dem_f, 2), np.nan)   # smooth, clipped contours

    ctx = dict(extent=extent, X=X, Y=Y, hs=hs, dem_s=dem_s, dist=DIST, prov=PROV)

    # clip classes to province if boundaries are available
    if DR is not None:
        for k in CL:
            CL[k] = np.where(DR > 0, CL[k], 0)

    ENS, DIS = {}, {}
    for s, p in RUNS:
        ENS[(s, p)], DIS[(s, p)] = ensemble(CL[('RF', s, p)], CL[('MaxEnt', s, p)])

    fig_current_models(CL[('RF',) + RUNS[0]], CL[('MaxEnt',) + RUNS[0]], ctx)
    fig_current(ENS[RUNS[0]], DIS[RUNS[0]], ctx)
    fut = {k: v for k, v in ENS.items() if k[0] != 'Current'}
    fig_scenarios(fut, {k: DIS[k] for k in fut}, ctx, 'Ensemble (RF + MaxEnt)')
    if PER_MODEL_FIGS:
        for m in MODELS:
            fig_scenarios({(s, p): CL[(m, s, p)] for s, p in fut}, None, ctx, f'{m}')

    # class-area table (% of province) for every run
    rows = []
    for (s, p), e in ENS.items():
        v = e[e > 0]
        rows.append([s, p] + [round(100 * (v == c).mean(), 2) for c in range(1, 6)])
    import pandas as pd
    df = pd.DataFrame(rows, columns=['Scenario', 'Period'] + CLASS_NAMES)
    df.to_csv(os.path.join(OUT_DIR, 'GWPZ_ensemble_class_area_pct.csv'), index=False)
    print(df.to_string(index=False))


if __name__ == '__main__':
    main()

  7 districts: Bugesera, Gatsibo, Kayonza, Kirehe, Ngoma, Nyagatare, Rwamagana
Loading DEM ...
  DEM range 0-2246 m, 0.2% of grid without data
saved /content/figures/Fig_GWPZ_current_RF_vs_MaxEnt.png
saved /content/figures/Fig_GWPZ_current_ensemble.png
saved /content/figures/Fig_GWPZ_scenarios_Ensemble.png
saved /content/figures/Fig_GWPZ_scenarios_RF.png
saved /content/figures/Fig_GWPZ_scenarios_MaxEnt.png
Scenario    Period  Very Low   Low  Moderate  High  Very High
 Current 1991-2020     41.38 35.04     12.40  6.78       4.40
 SSP1-26 2040-2059     47.37 35.47     10.20  5.07       1.88
 SSP1-26 2080-2099     47.47 35.83     10.10  4.87       1.73
 SSP2-45 2040-2059     34.30 41.49     13.24  6.87       4.10
 SSP2-45 2080-2099     34.53 41.38     13.18  6.85       4.06
 SSP5-85 2040-2059     47.50 36.34      9.91  4.67       1.59
 SSP5-85 2080-2099     47.81 38.05      9.28  3.78       1.08
